# Dhara.ai - run the pipeline on your own orthomosaic (GPU)

Use this notebook when you have a **real georeferenced orthomosaic GeoTIFF** (metric CRS such as UTM) or a large scene.
Runtime: *Runtime > Change runtime type > T4 GPU*.

Outputs are candidate layers (buildings, roads, vegetation, parcels, topology flags) plus a GeoPackage for QGIS/ArcGIS.

In [ ]:
!git clone https://github.com/shiv-codez/dhara-ai.git
%cd dhara-ai/backend
!pip -q install -r requirements.txt
!pip -q install torch torchvision timm git+https://github.com/ChaoningZhang/MobileSAM.git
!bash get_weights.sh

## 1. Upload the orthomosaic
It must be a GeoTIFF with a **projected (metric) CRS**. Re-project first with `gdalwarp -t_srs EPSG:32643 in.tif out.tif` if needed (pick the UTM zone of your area).

In [ ]:
from google.colab import files
up = files.upload()                      # choose your .tif
TIF = next(iter(up))
SCENE_ID, TITLE = "my_area", "My area"   # <- change

In [ ]:
import shutil, os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
from pathlib import Path
from dhara.config import Scene, Params, DATA_OUT
from dhara.georef import read_orthomosaic

out = DATA_OUT / SCENE_ID
out.mkdir(parents=True, exist_ok=True)
shutil.copy(TIF, out / "orthomosaic.tif")
rgb, tr, crs, tags = read_orthomosaic(out / "orthomosaic.tif")
print("size", rgb.shape, "| CRS", crs, "| GSD m/px", abs(tr.a))

## 2. Segment (SAM) + build vectors, parcels and topology checks
`points_per_side` higher = more objects found, slower. Tiles keep memory bounded on big images.

In [ ]:
import torch
from dhara.segment import MobileSamSegmenter
from dhara.pipeline import run_scene

params = Params(tile_size_px=1024, tile_overlap_px=128, points_per_side=32, points_per_batch=64)
seg = MobileSamSegmenter(params, device="cuda" if torch.cuda.is_available() else "cpu")
scene = Scene(id=SCENE_ID, title=TITLE, image="", origin_lonlat=(0, 0), gsd_m=abs(tr.a))   # georef comes from the GeoTIFF
manifest = run_scene(scene, params, segmenter=seg)
manifest["counts"], manifest["topology"]

## 3. Download everything (GeoPackage, GeoJSON layers, preview images)

In [ ]:
!cd {out}/web && zip -r /content/{SCENE_ID}_outputs.zip .
files.download(f"/content/{SCENE_ID}_outputs.zip")

## 4. Measure accuracy (only if you have reference polygons)
Load survey / hand-digitised reference polygons for the same area and compute real metrics.

In [ ]:
import geopandas as gpd
from dhara.metrics import polygon_scores, boundary_offset
gt = gpd.read_file("reference_buildings.geojson").to_crs(crs)     # <- your reference layer
pred = gpd.read_file(out / "web" / "buildings.geojson").to_crs(crs)
print(polygon_scores(pred, gt))
print(boundary_offset(pred, gt))